# 26 · The JND loss against the SSIM loss, on the two best models

The perceptual repo beside this one (`perceptual_jnd_loss.py`, its spec §3) holds a differentiable JND metric: the
visible error in just-noticeable differences, after the eye's contrast sensitivity, a brightness gain and masking.
This notebook asks whether training on it gives pictures that look better than training on the loss every model here
uses, `metrics.ssim_l1_loss` (84% luma SSIM, 16% RGB L1).

Two bodies, the best two in this folder:

- **told 24×2** - notebook 22's, what desktop-streamer runs as `upscale.onnx` (notebook 23);
- **fast q24** with its frozen 4×4 skip - notebook 24's drop-in for 60 fps on a GTX 1660 Ti.

Each is trained two ways on notebook 0's set, at seeds 0 and 1, with the shared recipe (`training.fit`): the same
data, Adam from 1e-3, batch 32 and the same halving schedule:

1. **SSIM loss** - notebook 22's and 24's own trainings, loaded from their run files; best epoch by SSIM-Y.
2. **JND loss** - `PerceptualJNDLoss` with the settings in `helpers/jnd_settings.json` (45 px a degree, 64 px tiles,
   d / (d + 1) a picture); best epoch by the same JND on the select split.

A judge that is also a loss favours the model trained on it: SSIM-Y and PSNR favour the first, JND and its MOS the
second. So both are also judged by two scores neither trained on - **LPIPS**, fitted to people's choices between
distorted patches, and **GMSD** - and by looking at them, on patches and on whole frames the host's stream made.

Every score is on notebook 0's validation split. Needs: `22_upscale_qp_slice_models.ipynb` and `24_upscale_fast.ipynb` (the SSIM-loss runs).

In [ ]:
"""Settings."""

import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from IPython.display import Image as Show, display
from PIL import Image
from torch.utils.data import DataLoader

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import degrade, judges, metrics, models, patches, sources, training
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 26
RUN_DIR = training.CHECKPOINT_DIR / "jnd_runs"                     # the JND-loss trainings, saved every epoch
SAMPLES = training.SAMPLE_DIR / "26_jnd" / "one_set"              # pictures, written to look at

QP_MAX = patches.QP_MAX
CONDITIONS = ("qp", "deblocked", "intra")
COLUMNS = patches.columns(CONDITIONS)
SEEDS = (0, 1)
LOSSES = ("SSIM loss", "JND loss")
# the two bodies: how to build each, and the run files of their SSIM-loss trainings (notebooks 22 and 24)
BODIES = {
    "told 24x2": {"build": lambda: models.ToldLuma(len(COLUMNS), 24, 2, 3),
                  "ssim_runs": training.CHECKPOINT_DIR / "qp_runs" / "told_qp_frame_seed{seed}.pt"},
    "fast q24": {"build": lambda: models.FastLuma(width=24, stem=4, blocks=1, skip="truncated"),
                 "ssim_runs": training.CHECKPOINT_DIR / "fast_runs" / "q24_frozen_skip_seed{seed}.pt"},
}

manifest = training.load_manifest()
SCALE = manifest["scale"]
trim = training.trimmer(manifest)
SAMPLES.mkdir(parents=True, exist_ok=True)
print("perceptual repo:", judges.PERCEPTUAL_DIR)

## 1. Data

Notebook 0's validation split, every patch with its slice QP / 51, loop filter and I frame, grouped by what it shows,
by line on P frames, and I frames.

In [ ]:
val_set = patches.PatchPairs(DATA_DIR, "val", manifest, augment=False, told=CONDITIONS)
val_loader = DataLoader(val_set, batch_size=128)
files = manifest["splits"]["val"]
labels = {name: np.load(DATA_DIR / files[name]) for name in ("kind", "encoder", "intra", "qp")}
intra = labels["intra"].astype(bool)
GROUPS = {"all": np.ones(len(val_set), bool),
          **{f"{kind}s" if kind != "video" else kind: labels["kind"] == k for k, kind in enumerate(manifest["kinds"])},
          **{f"{encoder} P": (labels["encoder"] == e) & ~intra for e, encoder in enumerate(manifest["encoders"])},
          "I frames": intra}
print("val: " + ", ".join(f"{name} {mask.sum()}" for name, mask in GROUPS.items()))

## 2. Models and the loss

The JND loss is `PerceptualJNDLoss` with `helpers/jnd_settings.json`: 45 px a degree (a desktop monitor at an arm's length), 64 px tiles, masking exponent 1.2,
tiles pooled at β = 16, the tile-mean colour as CIELAB ΔE76 (smooth, for training), and d / (d + 1) a picture,
averaged over the batch. It differs from the spec's defaults in the masking exponent (0.7) and the tile pooling
(β = 4), and in the curve from d to the MOS the judge reports (ρ 0.551 and d50 2.83, against 0.6 and 5). On a 128 px
patch that is 2×2 tiles, so β = 16 counts mostly the worst tile. Its masking is held constant under the gradient, so a
model cannot hide an error behind texture it adds. Bicubic's JND below says how much visible error there is to remove.

In [ ]:
def build(body, seed):
    torch.manual_seed(seed)
    return BODIES[body]["build"]().to(DEVICE)


class Bicubic(nn.Module):
    def forward(self, x, condition=None):
        return models.bicubic(x, SCALE)


jnd_loss = judges.jnd_loss()   # the training loss: d / (d + 1) a picture, the batch's mean


def jnd_score(prediction, target):
    """What the JND runs keep their best epoch by: the loss on the select pictures, clamped as shown."""
    return {"jnd": jnd_loss(prediction.clamp(0, 1), target)}


bicubic_scores = training.evaluate(Bicubic(), val_loader, trim, jnd_score)
print(f"bicubic: SSIM-Y {bicubic_scores['ssim']:.4f}, luma {bicubic_scores['y']:.2f} dB, "
      f"JND loss {bicubic_scores['jnd']:.4f}")

## 3. Train

The shared recipe for both losses (`training.fit`): the seed sets the starting weights and then the batches, Adam from
1e-3, the rate halved after 6 epochs without improvement, stop when a hundredth of it brings nothing. The SSIM-loss
runs are notebooks 22's and 24's own run files, which `fit` loads. The JND runs keep the epoch with the lowest JND on
the select split (an improvement is 1e-5, as for SSIM-Y). Each is saved to `checkpoints/jnd_runs/` after every epoch.

In [ ]:
def run_file(body, loss, seed):
    if loss == "SSIM loss":
        return Path(str(BODIES[body]["ssim_runs"]).format(seed=seed))
    return RUN_DIR / f"{body.replace(' ', '_')}_seed{seed}.pt"


def train(body, loss, seed):
    model = build(body, seed)
    recipe = ({} if loss == "SSIM loss" else
              dict(loss_fn=jnd_loss, select="jnd", scorer=jnd_score, lower_is_better=True, min_improvement=1e-5))
    history, best = training.fit(model, run_file(body, loss, seed), manifest, told=CONDITIONS, seed=seed, verbose=0,
                                 **recipe)
    print(f"{body:<10} {loss:<10} seed {seed}: best epoch {best['epoch']:3} of {len(history):3}, "
          f"{recipe.get('select', 'ssim')} {best['score']:.4f}")
    return model.eval(), history

In [ ]:
trained, histories = {}, {}
for seed in SEEDS:
    for body in BODIES:
        for loss in LOSSES:
            trained[body, loss, seed], histories[body, loss, seed] = train(body, loss, seed)

## 4. Scores

Every judge on every validation patch, the halo trimmed and the picture clamped as shown:

| judge | better | trained on by |
|---|---|---|
| SSIM-Y | higher | the SSIM loss (84% of it) |
| luma dB | higher | the SSIM loss's L1, in part |
| JND (d, CIEDE2000 for the tile means) | lower | the JND loss (ΔE76 for the tile means) |
| MOS predicted from that d (BT.500, 1-5) | higher | the JND loss, through d |
| LPIPS (AlexNet, v0.1) | lower | neither |
| GMSD | lower | neither |

A model's score is the mean of its two seeds. The networks add the same value to R, G and B, so U and V are the skip's
under either loss and are not shown.

In [ ]:
judge_jnd, lpips = judges.jnd_judge(), judges.LPIPS().to(DEVICE)
JUDGES = {"ssim": ("SSIM-Y", 1), "y": ("luma dB", 1), "jnd": ("JND", -1), "mos": ("MOS", 1),
          "lpips": ("LPIPS", -1), "gmsd": ("GMSD", -1)}


@torch.no_grad()
def per_patch(model):
    """Every judge on every validation patch, as arrays."""
    model.eval()
    out = {key: [] for key in JUDGES}
    for (lr, condition), hr in val_loader:
        lr, condition, hr = lr.to(DEVICE), condition.to(DEVICE), hr.to(DEVICE)
        prediction = trim(model(lr, condition)).clamp(0, 1)
        each, rated = metrics.per_image(prediction, hr), judge_jnd.evaluate(prediction, hr)
        for key, value in (("ssim", each["ssim"]), ("y", each["y"]), ("jnd", rated["jnd"]), ("mos", rated["mos"]),
                           ("lpips", lpips(prediction, hr)), ("gmsd", judges.gmsd(prediction, hr))):
            out[key].append(value.float().cpu())
    return {key: torch.cat(value).numpy() for key, value in out.items()}


scores = {"bicubic": per_patch(Bicubic())}
for key, model in trained.items():
    scores[key] = per_patch(model)


def arm(body, loss, key):
    """A model's score on every patch, the mean of its seeds."""
    return np.mean([scores[body, loss, seed][key] for seed in SEEDS], axis=0)


def print_group(name):
    mask = GROUPS[name]
    print(f"{name} ({mask.sum()} patches)")
    print(f"  {'':<22}" + "".join(f"{label:>9}" for label, _ in JUDGES.values()))
    print(f"  {'bicubic':<22}" + "".join(f"{scores['bicubic'][key][mask].mean():9.4f}" for key in JUDGES))
    for body in BODIES:
        for loss in LOSSES:
            print(f"  {body + ', ' + loss:<22}" + "".join(f"{arm(body, loss, key)[mask].mean():9.4f}" for key in JUDGES))


for name in GROUPS:
    print_group(name)
    print()

print("how far apart the two seeds land, all patches")
for body in BODIES:
    for loss in LOSSES:
        gaps = [abs(scores[body, loss, 0][key].mean() - scores[body, loss, 1][key].mean()) for key in JUDGES]
        print(f"  {body + ', ' + loss:<22}" + "".join(f"{gap:9.4f}" for gap in gaps))

for (body, loss, seed), model in trained.items():   # SSIM-Y and dB for the table of every notebook (notebook 29)
    if loss == "JND loss":
        training.record(NOTEBOOK, f"{body}, JND loss" + (f", seed {seed}" if seed else ""),
                        training.val_scores(lambda lr, condition: model(lr, condition[:, COLUMNS]), manifest))

Means hide how often: below, the share of patches on which the JND-loss model scores better than the SSIM-loss one of
the same body, by judge (each the mean of its two seeds; a tie counts for neither).

In [ ]:
print(f"{'':<34}" + "".join(f"{label:>9}" for label, _ in JUDGES.values()))
for body in BODIES:
    for name in ("all", *[name for name in GROUPS if name in ("screens", "photos", "video")]):
        mask = GROUPS[name]
        cells = [np.mean(sign * (arm(body, "JND loss", key) - arm(body, "SSIM loss", key))[mask] > 0)
                 for key, (_, sign) in JUDGES.items()]
        print(f"{body + ', ' + name:<34}" + "".join(f"{cell:9.0%}" for cell in cells))

## 5. Pictures

What the numbers cannot settle, the eye can. Each mosaic is one picture a row, at twice its size with no smoothing:
the truth, bicubic, then each body under the SSIM loss and the JND loss, all at seed 0 (not the better seed of either).

First the validation patches where the two told 24×2 models differ most (the mean luma difference between their
pictures): if the losses lead to pictures that look different, it is here. Then screens, at random.

In [ ]:
COLUMNS = ["truth", "bicubic", *[f"{body}, {loss}" for body in BODIES for loss in LOSSES]]


def to_uint8(x):
    return (x.clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()


def mosaic(columns, path, zoom=2, gap=6):
    """{name: uint8 (N, H, W, 3)} side by side, a picture a row, written as a PNG at `zoom` x (nearest)."""
    first = next(iter(columns.values()))
    n, h, w = first.shape[:3]
    canvas = np.full((n * (h * zoom + gap) - gap, len(columns) * (w * zoom + gap) - gap, 3), 255, np.uint8)
    for c, images in enumerate(columns.values()):
        for r, image in enumerate(images):
            y, x = r * (h * zoom + gap), c * (w * zoom + gap)
            canvas[y:y + h * zoom, x:x + w * zoom] = image.repeat(zoom, 0).repeat(zoom, 1)
    Image.fromarray(canvas).save(path)
    return path


@torch.no_grad()
def patch_pictures(indices):
    """{column: uint8 pictures} of these validation patches."""
    batch = [val_set[int(i)] for i in indices]
    lr = torch.stack([b[0][0] for b in batch]).to(DEVICE)
    condition = torch.stack([b[0][1] for b in batch]).to(DEVICE)
    hr = torch.stack([b[1] for b in batch])
    out = {"truth": to_uint8(hr), "bicubic": to_uint8(trim(models.bicubic(lr, SCALE)))}
    for body in BODIES:
        for loss in LOSSES:
            out[f"{body}, {loss}"] = to_uint8(trim(trained[body, loss, 0](lr, condition)))
    return out


@torch.no_grad()
def luma_gap(a, b):
    """Per patch, the mean absolute luma difference between two models' pictures (0-255)."""
    out = []
    for (lr, condition), _ in val_loader:
        lr, condition = lr.to(DEVICE), condition.to(DEVICE)
        ya, yb = (metrics.rgb_to_yuv(trim(m(lr, condition)).clamp(0, 1))[:, 0] for m in (a, b))
        out.append(255 * (ya - yb).abs().mean(dim=(1, 2)).cpu())
    return torch.cat(out).numpy()


gap = luma_gap(trained["told 24x2", "SSIM loss", 0], trained["told 24x2", "JND loss", 0])
print(f"told 24x2, SSIM against JND loss: mean luma difference {gap.mean():.2f} of 255 "
      f"(median {np.median(gap):.2f}, 99th percentile {np.quantile(gap, 0.99):.2f})")
most = np.argsort(gap)[::-1][:6]
screens = np.random.default_rng(0).choice(np.flatnonzero(GROUPS["screens"]), 6, replace=False)
print("columns:", " | ".join(COLUMNS))
for name, indices in (("differ_most", most), ("screens", screens)):
    print(f"{name}: " + ", ".join(f"{manifest['kinds'][labels['kind'][i]]} (slice QP {labels['qp'][i]:.0f})"
                                  for i in indices))
    display(Show(filename=str(mosaic(patch_pictures(indices), SAMPLES / f"patches_{name}.png"))))

Then whole frames, as the client sees them: the first validation clip of five groups of video, 1280×720 from its centre, frame
16 (a P frame), through the host's x264 line twice - at its default 0.46 bits a pixel (a 640×360 stream), where a still
screen settles near QP 0, and starved at 0.5 Mbit/s, where the codec's damage shows. The models run on the whole stream
frame with the client's 4 px of edge read again around it, told the frame's slice QP, loop filter and type as the client
tells them. Shown at 2× is the 320×180 window where the two told 24×2 models differ most: the truth and bicubic on the
left, then the told 24×2 (top) and the fast q24 (bottom) under the SSIM loss and the JND loss. Every whole frame is
written to `data/samples/26_jnd/one_set/frames/`, with what the stream did in `frames.json`.

In [ ]:
FRAME_TOPICS = ("video: desktop and software use", "video: creative software", "video: pc games",
                "video: animation", "video: camera footage")
STREAMS = {"default": None, "0.5mbps": 0.5}    # the host's 0.46 bits a pixel, and a starved 0.5 Mbit/s
FRAME, CLIP_FRAMES, SIZE, WINDOW, HALO = 16, 24, (1280, 720), (180, 320), 4
LAYOUT = [["truth", "told 24x2, SSIM loss", "told 24x2, JND loss"], ["bicubic", "fast q24, SSIM loss", "fast q24, JND loss"]]
FRAMES_DIR = SAMPLES / "frames"
FRAMES_DIR.mkdir(exist_ok=True)
clip_paths = {f"yt8m-videos/{path.parts[-2]}/{path.stem}": path for path in sources.files("yt8m-videos", "test")}
keys = sorted({s["key"] for s in manifest["splits"]["val"]["sources"]} & set(clip_paths))
topic_of = {s["key"]: s["group"] for s in manifest["splits"]["val"]["sources"]}


def first_clip(topic):
    """The first validation clip of a topic with CLIP_FRAMES frames of 1280x720: its key, those frames from its centre,
    and its frame rate."""
    width, height = SIZE
    for key in keys:
        if topic_of[key] != topic:
            continue
        frames, fps = sources.load_sequence(clip_paths[key])
        if len(frames) >= CLIP_FRAMES and frames.shape[1] >= height and frames.shape[2] >= width:
            top, left = (frames.shape[1] - height) // 4 * 2, (frames.shape[2] - width) // 4 * 2
            return key, frames[:CLIP_FRAMES, top:top + height, left:left + width], fps
    return None, None, None


def stream_frame(frames, fps, mbps):
    """(HR, LR, condition) of frame FRAME through the host's x264 line, at `mbps` (None: its default bits a pixel)."""
    bpp = degrade.STREAM_BPP if mbps is None else mbps * 1e6 / (SIZE[0] // SCALE * SIZE[1] // SCALE * fps)
    made = degrade.Degradation(scale=SCALE, bpp=bpp, fps=fps, filter="bicubic", deblock=False, encoder="x264")
    hr, lr, _, report = degrade.degrade(frames, made, with_frames=True)
    row = report[FRAME]
    condition = np.array([row["slice_qp"] / QP_MAX, row["deblocked"], row["type"] == "I"], np.float32)
    return hr[FRAME], lr[FRAME], condition, bpp


@torch.no_grad()
def upscale_frame(model, lr, condition):
    """A whole stream frame through a model, its edge pixels read again around it as the client does."""
    x = torch.from_numpy(np.pad(lr, ((HALO, HALO), (HALO, HALO), (0, 0)), mode="edge")).permute(2, 0, 1)[None]
    x = x.float().div(255).to(DEVICE)
    c = torch.from_numpy(condition)[None, :, None, None].to(DEVICE)
    out = model(x, c)[..., HALO * SCALE:-HALO * SCALE, HALO * SCALE:-HALO * SCALE]
    return to_uint8(out)[0]


def busiest_window(a, b):
    """The (top, left) of the WINDOW where two pictures' luma differs most, on a grid of quarter windows."""
    luma = np.array([0.2126, 0.7152, 0.0722])
    diff = torch.from_numpy(np.abs(a.astype(np.float32) @ luma - b.astype(np.float32) @ luma))[None, None]
    sums = torch.nn.functional.avg_pool2d(diff, WINDOW, stride=(WINDOW[0] // 4, WINDOW[1] // 4))[0, 0].numpy()
    y, x = np.unravel_index(np.argmax(sums), sums.shape)
    return y * WINDOW[0] // 4, x * WINDOW[1] // 4


def grid(pictures, layout, path, zoom=2, gap=8):
    """Pictures placed as `layout` (rows of names), at `zoom` x with no smoothing, written as a PNG."""
    h, w = next(iter(pictures.values())).shape[:2]
    canvas = np.full((len(layout) * (h * zoom + gap) - gap, len(layout[0]) * (w * zoom + gap) - gap, 3), 255, np.uint8)
    for r, row in enumerate(layout):
        for c, name in enumerate(row):
            y, x = r * (h * zoom + gap), c * (w * zoom + gap)
            canvas[y:y + h * zoom, x:x + w * zoom] = pictures[name].repeat(zoom, 0).repeat(zoom, 1)
    Image.fromarray(canvas).save(path)
    return path


facts = {}
print("each grid: " + " | ".join(LAYOUT[0]) + "  /  " + " | ".join(LAYOUT[1]))
for topic in FRAME_TOPICS:
    key, frames, fps = first_clip(topic)
    if key is None:
        print(f"{topic}: no clip of 1280x720 and {CLIP_FRAMES} frames")
        continue
    for stream, mbps in STREAMS.items():
        hr, lr, condition, bpp = stream_frame(frames, fps, mbps)
        pictures = {"truth": hr, "bicubic": upscale_frame(Bicubic(), lr, condition)}
        for body in BODIES:
            for loss in LOSSES:
                pictures[f"{body}, {loss}"] = upscale_frame(trained[body, loss, 0], lr, condition)
        name = f"{topic.split(": ")[1].replace(' ', '_')}_{stream}"
        for label, picture in pictures.items():
            Image.fromarray(picture).save(FRAMES_DIR / f"{name}_{label.replace(', ', '_').replace(' ', '_')}.png")
        facts[name] = {"topic": topic, "clip": key, "frame": FRAME, "stream": stream, "bpp": round(bpp, 4),
                       "slice_qp": round(float(condition[0] * QP_MAX)), "deblocked": bool(condition[1]),
                       "intra": bool(condition[2]), "encoder": "x264"}
        top, left = busiest_window(pictures["told 24x2, SSIM loss"], pictures["told 24x2, JND loss"])
        window = {label: p[top:top + WINDOW[0], left:left + WINDOW[1]] for label, p in pictures.items()}
        print(f"{topic}, {stream} ({bpp:.3f} bpp): slice QP {facts[name]['slice_qp']}, window at ({left}, {top})")
        display(Show(filename=str(grid(window, LAYOUT, SAMPLES / f"frame_{name}.png"))))
(FRAMES_DIR / "frames.json").write_text(json.dumps(facts, indent=1), encoding="utf-8")

## 6. What kind of difference

The scores say how much; this says what. On the validation patches, both seeds of each model: how much of the truth's
fine detail it keeps (the mean |Laplacian| of luma, over the truth's), its mean luma bias, and its luma error at the
truth's strongest edges (the top fifth of |Laplacian| in each patch) and everywhere else, in 8-bit levels. Then on the
whole frames above, where the stream's grid is known, how much larger a step in luma is across a seam than anywhere
else (1 is no seam): between LR pixels (every 2 HR px), between the fast q24's 2×2 LR cells (every 4) and between the
codec's macroblocks (every 16).

In [ ]:
LAPLACE = torch.tensor([[0.0, 1, 0], [1, -4, 1], [0, 1, 0]], device=DEVICE)[None, None]


@torch.no_grad()
def character(model):
    """Detail kept, luma bias, and the error at edges and elsewhere, over the validation patches."""
    total = {"detail": 0.0, "truth_detail": 0.0, "bias": 0.0, "edges": 0.0, "flat": 0.0}
    for (lr, condition), hr in val_loader:
        lr, condition, hr = lr.to(DEVICE), condition.to(DEVICE), hr.to(DEVICE)
        truth = metrics.rgb_to_yuv(hr)[:, :1]
        picture = metrics.rgb_to_yuv(trim(model(lr, condition)).clamp(0, 1))[:, :1]
        lt, lp = (torch.nn.functional.conv2d(y, LAPLACE).abs() for y in (truth, picture))
        edges = (lt > lt.flatten(1).quantile(0.8, dim=1)[:, None, None, None]).float()
        error = (picture - truth).abs()[..., 1:-1, 1:-1]
        n = len(hr)
        total["detail"] += lp.mean().item() * n
        total["truth_detail"] += lt.mean().item() * n
        total["bias"] += 255 * (picture - truth).mean().item() * n
        total["edges"] += 255 * ((error * edges).sum() / edges.sum()).item() * n
        total["flat"] += 255 * ((error * (1 - edges)).sum() / (1 - edges).sum()).item() * n
    return {"detail": total["detail"] / total["truth_detail"], **{k: total[k] / len(val_set) for k in ("bias", "edges", "flat")}}


def seam(y, period):
    """The mean step in luma across seams every `period` px, over the mean step elsewhere, both axes."""
    out = []
    for a in (y, y.T):
        step = np.abs(np.diff(a, axis=1))
        on = np.arange(step.shape[1]) % period == period - 1
        out.append(step[:, on].mean() / step[:, ~on].mean())
    return float(np.mean(out))


luma = np.array([0.2126, 0.7152, 0.0722])
rows = {"bicubic": [Bicubic()], **{f"{body}, {loss}": [trained[body, loss, seed] for seed in SEEDS]
                                   for body in BODIES for loss in LOSSES}}
print(f"{'':<22}{'detail kept':>12}{'luma bias':>10}{'at edges':>10}{'elsewhere':>10}"
      f"{'seam 2 px':>11}{'seam 4 px':>11}{'seam 16 px':>12}")
for name in ["truth", *rows]:
    if name == "truth":
        cells = ["", "", "", ""]
    else:
        each = [character(model) for model in rows[name]]
        cells = [f"{np.mean([c[key] for c in each]):.3f}" for key in ("detail", "bias", "edges", "flat")]
    label = name.replace(", ", "_").replace(" ", "_")
    frames = [np.asarray(Image.open(FRAMES_DIR / f"{frame}_{label}.png")).astype(np.float64) @ luma for frame in facts]
    seams = [np.mean([seam(y, period) for y in frames]) for period in (2, 4, 16)]
    print(f"{name:<22}" + "".join(f"{cell:>{width}}" for cell, width in zip(cells, (12, 10, 10, 10)))
          + "".join(f"{s:{width}.3f}" for s, width in zip(seams, (11, 11, 12))))

## 7. What it says

*(written after the run)*